In [14]:
import pandas as pd
import numpy as np

In [15]:
PODS = ["Machine Learning",
        "Deep Learning",
        "Python",
        "Data Visualization",
        "R Studio",
        "GitHub / Open Science",
        "Diffusion Neuroimaging",
        "TMS",
        "EEG",
        "fMRI",
        "Behavioral Differences",
        "Spatial Navigation",
        "Taste Perception",
        "Auditory Perception",
        "Aging",
        "Depression",
        "Schizophernia",
        "ADHD",
        "Addiction",
        "Linguistics / LLMs",
        "Higher Order Reasoning / Decision"]

DIRECTORY = "/Users/jowanglin/Downloads"
FILE_NAME =  "BrainHack School Taiwan-Singapore 2026 Pods and Keywords Survey (Responses).xlsx"
SHEET_NAME = "Form Responses 1"

In [17]:
def mapper(col):
    if "Time" in col:
        return "time"
    elif "Student" in col:
        return "name"
    elif "expertise" in col:
        return "expertise"
    elif "modality" in col:
        return "modality"
    elif "keywords" in col:
        return "keywords"
    elif "project in mind" in col:
        return "project"
    elif "complete" in col:
        return "complete"
    else:
        return col.lower()

df = pd.read_excel(f"{DIRECTORY}/{FILE_NAME}", sheet_name=SHEET_NAME)
df = df.rename(mapper=mapper, axis=1).iloc[:, :-2]
# display(df)


In [18]:
# for > 3 interest keywords, filter first to keep the ones that match existing pods
# if still > 3, randomly shuffle and pick the first 3 elements
def shuffle_and_pick(k):
    k = np.asarray([i for i in k if i in PODS])
    if len(k) > 3:
        np.random.shuffle(k)
        k = k[:3]
    return k

names, keywords = df["name"], df["keywords"]
idx_to_name = {idx: name for idx, name in enumerate(names)}

keywords = keywords.map(lambda s: s.split(","))
keywords = keywords.map(lambda s: ["fMRI" if "MRI" in i else i for i in s])
keywords = keywords.map(lambda s: [i.strip() for i in s])
_keywords = keywords.copy()
keywords = [shuffle_and_pick(k) if len(k) > 3 else np.asarray(k) for k in keywords]
keywords = [np.pad(k, (0, 3 - len(k))) for k in keywords]
keywords = np.stack(keywords, axis=0)
assert len(names) == keywords.shape[0]
print(keywords.shape)

(26, 3)


In [19]:
pods_assign = [np.where(np.isin(keywords, p))[0] for p in PODS]
pods_assign = [[idx_to_name[idx] for idx in indices] for indices in pods_assign]
table = pd.DataFrame(data=pods_assign)
table.insert(0, "pods", PODS)
# display(table)

In [20]:
keywords_new = [[k for k in sublist if k not in PODS] for sublist in _keywords]
display(keywords_new)

[['Aggression', 'antisocial behaviours', 'emotional regulation', 'crime'],
 [],
 ['dyslexia'],
 [],
 ['Visual Perception', 'Attention'],
 ['ASD and socio-cognition function'],
 [],
 ['Emotion / Affective Neuroscience'],
 [],
 [],
 [],
 [],
 ['Motor learning'],
 ['matlab (I think I also need to know more about this......)'],
 [],
 [],
 ['CNN'],
 [],
 [],
 ['Bipolar Disorder / Circadian Rhythms'],
 ['Sleep or infant development'],
 [],
 [],
 [],
 ['MEG', 'Autism'],
 []]

In [21]:
# add emotion and ASD (autism)
def checker(key: str, lst):
    if key == "emotion":
        if any([key in l.lower() for l in lst]):
            return True
        else:
            return False
    elif key == "autism":
        def helper(l):
            if key in l.lower() or "ASD" in l:
                return True
            else:
                return False
        return True if any([helper(l) for l in lst]) else False

emo = np.where([checker("emotion", k) for k in keywords_new])[0]
asd = np.where([checker("autism", k) for k in keywords_new])[0]

max_pad = table.shape[1] - 1
emo = np.pad([idx_to_name[idx] for idx in emo],
             pad_width=(0, max_pad - len(emo)),
             constant_values=(np.nan,))
asd = np.pad([idx_to_name[idx] for idx in asd],
             pad_width=(0, max_pad - len(asd)),
             constant_values=(np.nan,))

to_concat = pd.DataFrame(data=np.stack([emo, asd]))
to_concat.insert(0, "pods", ["Emotion", "Autism"])
table = pd.concat([table, to_concat])
table = table.reset_index(drop=True)
# display(table)


In [22]:
table.to_excel(f"{DIRECTORY}/pods_assigned.xlsx", index=False)